# 🎬 AI 자동 비디오 편집기

이미지, 영상, 음악을 업로드하면 자동으로 분석하여 뮤직비디오/영화 스타일의 결과물을 생성합니다.

## 주요 기능
- 얼굴 감지 및 자동 줌인/줌아웃
- 음악 비트 분석 및 싱크
- 다양한 영화적 카메라 기법 적용
- 전문적인 트랜지션 효과

## 1. 필요한 라이브러리 설치

In [ ]:
!pip install moviepy==1.0.3
!pip install librosa
!pip install opencv-python-headless
!pip install mediapipe
!pip install numpy
!pip install pillow
!pip install scipy
!apt-get install -y ffmpeg

## 2. 라이브러리 임포트 및 초기 설정

In [ ]:
import os
import cv2
import numpy as np
import librosa
import mediapipe as mp
from PIL import Image
from google.colab import files
from moviepy.editor import (
    VideoFileClip, ImageClip, AudioFileClip, CompositeVideoClip,
    concatenate_videoclips, vfx, CompositeAudioClip
)
from moviepy.video.fx.all import crop, resize
from scipy.signal import find_peaks
import random
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

print("✅ 모든 라이브러리가 성공적으로 로드되었습니다!")

## 3. 핵심 클래스 및 설정 정의

In [ ]:
class CameraTechnique(Enum):
    """영화/드라마 카메라 기법"""
    ZOOM_IN = "zoom_in"           # 줌인 (클로즈업)
    ZOOM_OUT = "zoom_out"         # 줌아웃 (와이드샷)
    PAN_LEFT = "pan_left"         # 좌로 패닝
    PAN_RIGHT = "pan_right"       # 우로 패닝
    TILT_UP = "tilt_up"           # 위로 틸트
    TILT_DOWN = "tilt_down"       # 아래로 틸트
    DOLLY_IN = "dolly_in"         # 돌리 인 (부드러운 전진)
    DOLLY_OUT = "dolly_out"       # 돌리 아웃 (부드러운 후진)
    SHAKE = "shake"               # 핸드헬드 효과
    STATIC = "static"             # 고정샷
    KEN_BURNS = "ken_burns"       # 켄 번스 효과 (사진용)
    FACE_TRACK = "face_track"     # 얼굴 추적 줌


class TransitionType(Enum):
    """트랜지션 종류"""
    CUT = "cut"                   # 컷 (즉시 전환)
    FADE = "fade"                 # 페이드
    CROSSFADE = "crossfade"       # 크로스페이드
    WIPE_LEFT = "wipe_left"       # 좌측 와이프
    WIPE_RIGHT = "wipe_right"     # 우측 와이프
    ZOOM_TRANSITION = "zoom_trans" # 줌 트랜지션
    FLASH = "flash"               # 플래시 (화이트아웃)


@dataclass
class MediaInfo:
    """미디어 정보 저장"""
    filepath: str
    media_type: str  # 'image', 'video', 'audio'
    duration: float = 0.0
    width: int = 0
    height: int = 0
    faces: List[Dict] = None
    has_motion: bool = False
    dominant_colors: List[Tuple] = None


@dataclass
class BeatInfo:
    """음악 비트 정보"""
    beat_times: List[float]       # 비트 시점들
    tempo: float                   # BPM
    strong_beats: List[float]     # 강한 비트 시점
    sections: List[Tuple[float, float]]  # 구간 (시작, 끝)
    total_duration: float


@dataclass 
class EditDecision:
    """편집 결정 사항"""
    media_index: int
    start_time: float
    end_time: float
    camera_technique: CameraTechnique
    transition_in: TransitionType
    transition_out: TransitionType
    zoom_target: Optional[Dict] = None  # 줌 타겟 (얼굴 위치 등)


# 설정값
class Config:
    OUTPUT_WIDTH = 1920
    OUTPUT_HEIGHT = 1080
    OUTPUT_FPS = 30
    MIN_CLIP_DURATION = 1.5  # 최소 클립 길이 (초)
    MAX_CLIP_DURATION = 8.0  # 최대 클립 길이 (초)
    DEFAULT_IMAGE_DURATION = 4.0  # 이미지 기본 표시 시간
    TRANSITION_DURATION = 0.5  # 트랜지션 길이
    ZOOM_INTENSITY = 1.3  # 줌 강도 (1.0 = 변화없음)
    FACE_ZOOM_INTENSITY = 1.5  # 얼굴 줌 강도

print("✅ 설정 완료!")

## 4. 얼굴 감지 시스템

In [ ]:
class FaceDetector:
    """MediaPipe를 이용한 얼굴 감지"""
    
    def __init__(self):
        self.mp_face_detection = mp.solutions.face_detection
        self.face_detection = self.mp_face_detection.FaceDetection(
            model_selection=1,  # 0: 2m 이내, 1: 5m 이내
            min_detection_confidence=0.5
        )
    
    def detect_faces(self, image: np.ndarray) -> List[Dict]:
        """
        이미지에서 얼굴 감지
        Returns: [{'bbox': (x, y, w, h), 'confidence': float, 'center': (cx, cy)}]
        """
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = self.face_detection.process(rgb_image)
        
        faces = []
        if results.detections:
            h, w = image.shape[:2]
            for detection in results.detections:
                bbox = detection.location_data.relative_bounding_box
                x = int(bbox.xmin * w)
                y = int(bbox.ymin * h)
                face_w = int(bbox.width * w)
                face_h = int(bbox.height * h)
                
                # 중심점 계산
                center_x = x + face_w // 2
                center_y = y + face_h // 2
                
                faces.append({
                    'bbox': (x, y, face_w, face_h),
                    'confidence': detection.score[0],
                    'center': (center_x, center_y),
                    'relative_center': (center_x / w, center_y / h),
                    'relative_size': (face_w / w, face_h / h)
                })
        
        # 신뢰도 순으로 정렬
        faces.sort(key=lambda x: x['confidence'], reverse=True)
        return faces
    
    def detect_faces_in_video(self, video_path: str, sample_rate: int = 30) -> List[List[Dict]]:
        """비디오에서 샘플링하여 얼굴 감지"""
        cap = cv2.VideoCapture(video_path)
        faces_timeline = []
        frame_count = 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            if frame_count % sample_rate == 0:
                faces = self.detect_faces(frame)
                faces_timeline.append({
                    'frame': frame_count,
                    'time': frame_count / cap.get(cv2.CAP_PROP_FPS),
                    'faces': faces
                })
            
            frame_count += 1
        
        cap.release()
        return faces_timeline


# 전역 얼굴 감지기 인스턴스
face_detector = FaceDetector()
print("✅ 얼굴 감지 시스템 초기화 완료!")

## 5. 음악 분석 시스템

In [ ]:
class MusicAnalyzer:
    """음악 비트 및 구조 분석"""
    
    def analyze(self, audio_path: str) -> BeatInfo:
        """음악 파일 분석"""
        print(f"🎵 음악 분석 중: {audio_path}")
        
        # 오디오 로드
        y, sr = librosa.load(audio_path)
        duration = librosa.get_duration(y=y, sr=sr)
        
        # 템포 및 비트 감지
        tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
        beat_times = librosa.frames_to_time(beat_frames, sr=sr).tolist()
        
        # 강한 비트 감지 (onset strength 기반)
        onset_env = librosa.onset.onset_strength(y=y, sr=sr)
        peaks, properties = find_peaks(onset_env, height=np.mean(onset_env) * 1.5, distance=sr//10)
        strong_beat_frames = peaks
        strong_beats = librosa.frames_to_time(strong_beat_frames, sr=sr).tolist()
        
        # 구간 분석 (verse, chorus 등)
        sections = self._detect_sections(y, sr, duration)
        
        # numpy float를 일반 float로 변환
        if hasattr(tempo, '__iter__'):
            tempo = float(tempo[0]) if len(tempo) > 0 else 120.0
        else:
            tempo = float(tempo)
        
        print(f"  - 템포: {tempo:.1f} BPM")
        print(f"  - 총 비트 수: {len(beat_times)}")
        print(f"  - 강한 비트 수: {len(strong_beats)}")
        print(f"  - 구간 수: {len(sections)}")
        
        return BeatInfo(
            beat_times=beat_times,
            tempo=tempo,
            strong_beats=strong_beats,
            sections=sections,
            total_duration=duration
        )
    
    def _detect_sections(self, y, sr, duration) -> List[Tuple[float, float]]:
        """음악 구간 감지 (간단한 에너지 기반)"""
        # RMS 에너지 계산
        rms = librosa.feature.rms(y=y)[0]
        
        # 에너지 변화가 큰 지점 찾기
        rms_diff = np.abs(np.diff(rms))
        threshold = np.mean(rms_diff) + np.std(rms_diff)
        change_points = np.where(rms_diff > threshold)[0]
        change_times = librosa.frames_to_time(change_points, sr=sr)
        
        # 구간 생성
        sections = []
        prev_time = 0.0
        
        for t in change_times:
            if t - prev_time > 5.0:  # 최소 5초 이상 구간
                sections.append((prev_time, t))
                prev_time = t
        
        # 마지막 구간
        if duration - prev_time > 2.0:
            sections.append((prev_time, duration))
        
        # 구간이 너무 적으면 균등 분할
        if len(sections) < 3:
            section_length = duration / 4
            sections = [(i * section_length, (i + 1) * section_length) for i in range(4)]
        
        return sections


music_analyzer = MusicAnalyzer()
print("✅ 음악 분석 시스템 초기화 완료!")

## 6. 카메라 효과 엔진

In [ ]:
class CameraEffectEngine:
    """영화적 카메라 효과 적용"""
    
    def __init__(self, config: Config = Config()):
        self.config = config
    
    def apply_effect(self, clip, technique: CameraTechnique, 
                     face_info: Optional[Dict] = None) -> VideoFileClip:
        """클립에 카메라 효과 적용"""
        
        if technique == CameraTechnique.ZOOM_IN:
            return self._zoom_in(clip, face_info)
        elif technique == CameraTechnique.ZOOM_OUT:
            return self._zoom_out(clip, face_info)
        elif technique == CameraTechnique.PAN_LEFT:
            return self._pan(clip, direction='left')
        elif technique == CameraTechnique.PAN_RIGHT:
            return self._pan(clip, direction='right')
        elif technique == CameraTechnique.TILT_UP:
            return self._tilt(clip, direction='up')
        elif technique == CameraTechnique.TILT_DOWN:
            return self._tilt(clip, direction='down')
        elif technique == CameraTechnique.DOLLY_IN:
            return self._dolly(clip, direction='in')
        elif technique == CameraTechnique.DOLLY_OUT:
            return self._dolly(clip, direction='out')
        elif technique == CameraTechnique.SHAKE:
            return self._handheld_shake(clip)
        elif technique == CameraTechnique.KEN_BURNS:
            return self._ken_burns(clip, face_info)
        elif technique == CameraTechnique.FACE_TRACK:
            return self._face_track_zoom(clip, face_info)
        else:
            return clip
    
    def _zoom_in(self, clip, face_info=None):
        """줌인 효과 - 와이드에서 클로즈업으로"""
        duration = clip.duration
        w, h = clip.size
        
        # 줌 타겟 설정 (얼굴이 있으면 얼굴 중심으로)
        if face_info:
            target_x = face_info['relative_center'][0]
            target_y = face_info['relative_center'][1]
        else:
            target_x, target_y = 0.5, 0.5
        
        def zoom_effect(get_frame, t):
            progress = t / duration
            # 1.0에서 시작해서 ZOOM_INTENSITY까지 확대
            scale = 1.0 + (Config.ZOOM_INTENSITY - 1.0) * progress
            
            frame = get_frame(t)
            fh, fw = frame.shape[:2]
            
            # 새 크기 계산
            new_w = int(fw * scale)
            new_h = int(fh * scale)
            
            # 크롭 영역 계산 (타겟 중심으로)
            center_x = int(target_x * new_w)
            center_y = int(target_y * new_h)
            
            x1 = max(0, center_x - fw // 2)
            y1 = max(0, center_y - fh // 2)
            x2 = min(new_w, x1 + fw)
            y2 = min(new_h, y1 + fh)
            
            # 경계 조정
            if x2 - x1 < fw:
                x1 = max(0, x2 - fw)
            if y2 - y1 < fh:
                y1 = max(0, y2 - fh)
            
            # 리사이즈 후 크롭
            resized = cv2.resize(frame, (new_w, new_h))
            cropped = resized[y1:y1+fh, x1:x1+fw]
            
            return cropped
        
        return clip.fl(zoom_effect)
    
    def _zoom_out(self, clip, face_info=None):
        """줌아웃 효과 - 클로즈업에서 와이드로"""
        duration = clip.duration
        w, h = clip.size
        
        if face_info:
            target_x = face_info['relative_center'][0]
            target_y = face_info['relative_center'][1]
        else:
            target_x, target_y = 0.5, 0.5
        
        def zoom_effect(get_frame, t):
            progress = t / duration
            # ZOOM_INTENSITY에서 시작해서 1.0으로 축소
            scale = Config.ZOOM_INTENSITY - (Config.ZOOM_INTENSITY - 1.0) * progress
            
            frame = get_frame(t)
            fh, fw = frame.shape[:2]
            
            new_w = int(fw * scale)
            new_h = int(fh * scale)
            
            center_x = int(target_x * new_w)
            center_y = int(target_y * new_h)
            
            x1 = max(0, center_x - fw // 2)
            y1 = max(0, center_y - fh // 2)
            x2 = min(new_w, x1 + fw)
            y2 = min(new_h, y1 + fh)
            
            if x2 - x1 < fw:
                x1 = max(0, x2 - fw)
            if y2 - y1 < fh:
                y1 = max(0, y2 - fh)
            
            resized = cv2.resize(frame, (new_w, new_h))
            cropped = resized[y1:y1+fh, x1:x1+fw]
            
            return cropped
        
        return clip.fl(zoom_effect)
    
    def _pan(self, clip, direction='left'):
        """패닝 효과"""
        duration = clip.duration
        w, h = clip.size
        
        # 패닝을 위해 먼저 클립을 약간 확대
        scale = 1.2
        
        def pan_effect(get_frame, t):
            progress = t / duration
            frame = get_frame(t)
            fh, fw = frame.shape[:2]
            
            new_w = int(fw * scale)
            new_h = int(fh * scale)
            resized = cv2.resize(frame, (new_w, new_h))
            
            # 이동량 계산
            max_offset = new_w - fw
            
            if direction == 'left':
                x_offset = int(max_offset * (1 - progress))
            else:  # right
                x_offset = int(max_offset * progress)
            
            y_offset = (new_h - fh) // 2
            
            cropped = resized[y_offset:y_offset+fh, x_offset:x_offset+fw]
            return cropped
        
        return clip.fl(pan_effect)
    
    def _tilt(self, clip, direction='up'):
        """틸트 효과"""
        duration = clip.duration
        scale = 1.2
        
        def tilt_effect(get_frame, t):
            progress = t / duration
            frame = get_frame(t)
            fh, fw = frame.shape[:2]
            
            new_w = int(fw * scale)
            new_h = int(fh * scale)
            resized = cv2.resize(frame, (new_w, new_h))
            
            max_offset = new_h - fh
            x_offset = (new_w - fw) // 2
            
            if direction == 'up':
                y_offset = int(max_offset * (1 - progress))
            else:  # down
                y_offset = int(max_offset * progress)
            
            cropped = resized[y_offset:y_offset+fh, x_offset:x_offset+fw]
            return cropped
        
        return clip.fl(tilt_effect)
    
    def _dolly(self, clip, direction='in'):
        """돌리 효과 (부드러운 줌)"""
        duration = clip.duration
        
        def dolly_effect(get_frame, t):
            # 이징 함수로 부드러운 움직임
            progress = t / duration
            eased_progress = self._ease_in_out(progress)
            
            if direction == 'in':
                scale = 1.0 + (0.15 * eased_progress)
            else:
                scale = 1.15 - (0.15 * eased_progress)
            
            frame = get_frame(t)
            fh, fw = frame.shape[:2]
            
            new_w = int(fw * scale)
            new_h = int(fh * scale)
            resized = cv2.resize(frame, (new_w, new_h))
            
            x_offset = (new_w - fw) // 2
            y_offset = (new_h - fh) // 2
            
            cropped = resized[y_offset:y_offset+fh, x_offset:x_offset+fw]
            return cropped
        
        return clip.fl(dolly_effect)
    
    def _handheld_shake(self, clip):
        """핸드헬드 카메라 흔들림 효과"""
        duration = clip.duration
        
        # 미리 계산된 흔들림 값
        np.random.seed(42)
        num_samples = int(duration * 30)
        shake_x = np.cumsum(np.random.randn(num_samples) * 2)
        shake_y = np.cumsum(np.random.randn(num_samples) * 2)
        shake_x = shake_x - np.mean(shake_x)
        shake_y = shake_y - np.mean(shake_y)
        shake_x = np.clip(shake_x, -15, 15)
        shake_y = np.clip(shake_y, -15, 15)
        
        scale = 1.1
        
        def shake_effect(get_frame, t):
            frame_idx = min(int(t * 30), len(shake_x) - 1)
            dx = int(shake_x[frame_idx])
            dy = int(shake_y[frame_idx])
            
            frame = get_frame(t)
            fh, fw = frame.shape[:2]
            
            new_w = int(fw * scale)
            new_h = int(fh * scale)
            resized = cv2.resize(frame, (new_w, new_h))
            
            x_offset = (new_w - fw) // 2 + dx
            y_offset = (new_h - fh) // 2 + dy
            
            x_offset = max(0, min(new_w - fw, x_offset))
            y_offset = max(0, min(new_h - fh, y_offset))
            
            cropped = resized[y_offset:y_offset+fh, x_offset:x_offset+fw]
            return cropped
        
        return clip.fl(shake_effect)
    
    def _ken_burns(self, clip, face_info=None):
        """켄 번스 효과 (사진에 동적 움직임)"""
        duration = clip.duration
        
        # 랜덤하게 시작/끝 위치 결정
        if face_info:
            # 얼굴이 있으면 얼굴 중심으로 줌인
            start_scale = 1.0
            end_scale = Config.FACE_ZOOM_INTENSITY
            target_x = face_info['relative_center'][0]
            target_y = face_info['relative_center'][1]
        else:
            # 랜덤 방향
            start_scale = random.choice([1.0, 1.3])
            end_scale = 1.3 if start_scale == 1.0 else 1.0
            target_x = random.uniform(0.3, 0.7)
            target_y = random.uniform(0.3, 0.7)
        
        def kb_effect(get_frame, t):
            progress = t / duration
            eased = self._ease_in_out(progress)
            
            scale = start_scale + (end_scale - start_scale) * eased
            
            frame = get_frame(t)
            fh, fw = frame.shape[:2]
            
            new_w = int(fw * scale)
            new_h = int(fh * scale)
            resized = cv2.resize(frame, (new_w, new_h))
            
            center_x = int(target_x * new_w)
            center_y = int(target_y * new_h)
            
            x1 = max(0, center_x - fw // 2)
            y1 = max(0, center_y - fh // 2)
            
            if x1 + fw > new_w:
                x1 = new_w - fw
            if y1 + fh > new_h:
                y1 = new_h - fh
            
            cropped = resized[y1:y1+fh, x1:x1+fw]
            return cropped
        
        return clip.fl(kb_effect)
    
    def _face_track_zoom(self, clip, face_info):
        """얼굴 추적 줌 - 얼굴에 확실하게 포커스"""
        if not face_info:
            return self._zoom_in(clip, None)
        
        duration = clip.duration
        target_x = face_info['relative_center'][0]
        target_y = face_info['relative_center'][1]
        face_size = max(face_info['relative_size'])
        
        # 얼굴 크기에 따라 줌 강도 조절
        zoom_factor = min(Config.FACE_ZOOM_INTENSITY, 1.0 / (face_size * 2))
        zoom_factor = max(1.3, zoom_factor)  # 최소 1.3배
        
        def face_zoom_effect(get_frame, t):
            progress = t / duration
            eased = self._ease_in_out(progress)
            
            scale = 1.0 + (zoom_factor - 1.0) * eased
            
            frame = get_frame(t)
            fh, fw = frame.shape[:2]
            
            new_w = int(fw * scale)
            new_h = int(fh * scale)
            resized = cv2.resize(frame, (new_w, new_h))
            
            center_x = int(target_x * new_w)
            center_y = int(target_y * new_h)
            
            x1 = max(0, center_x - fw // 2)
            y1 = max(0, center_y - fh // 2)
            
            if x1 + fw > new_w:
                x1 = new_w - fw
            if y1 + fh > new_h:
                y1 = new_h - fh
            
            cropped = resized[y1:y1+fh, x1:x1+fw]
            return cropped
        
        return clip.fl(face_zoom_effect)
    
    def _ease_in_out(self, t):
        """부드러운 이징 함수"""
        return t * t * (3 - 2 * t)


camera_engine = CameraEffectEngine()
print("✅ 카메라 효과 엔진 초기화 완료!")

## 7. 트랜지션 엔진

In [ ]:
class TransitionEngine:
    """전문적인 트랜지션 효과"""
    
    def apply_transition(self, clip1, clip2, transition_type: TransitionType, 
                         duration: float = 0.5):
        """두 클립 사이에 트랜지션 적용"""
        
        if transition_type == TransitionType.CUT:
            return concatenate_videoclips([clip1, clip2])
        
        elif transition_type == TransitionType.CROSSFADE:
            return self._crossfade(clip1, clip2, duration)
        
        elif transition_type == TransitionType.FADE:
            return self._fade_through_black(clip1, clip2, duration)
        
        elif transition_type == TransitionType.FLASH:
            return self._flash_transition(clip1, clip2, duration)
        
        elif transition_type in [TransitionType.WIPE_LEFT, TransitionType.WIPE_RIGHT]:
            direction = 'left' if transition_type == TransitionType.WIPE_LEFT else 'right'
            return self._wipe(clip1, clip2, direction, duration)
        
        elif transition_type == TransitionType.ZOOM_TRANSITION:
            return self._zoom_transition(clip1, clip2, duration)
        
        else:
            return concatenate_videoclips([clip1, clip2])
    
    def _crossfade(self, clip1, clip2, duration):
        """크로스페이드"""
        clip1 = clip1.crossfadeout(duration)
        clip2 = clip2.crossfadein(duration)
        clip2 = clip2.set_start(clip1.duration - duration)
        return CompositeVideoClip([clip1, clip2])
    
    def _fade_through_black(self, clip1, clip2, duration):
        """블랙을 통한 페이드"""
        clip1 = clip1.fadeout(duration / 2)
        clip2 = clip2.fadein(duration / 2)
        return concatenate_videoclips([clip1, clip2])
    
    def _flash_transition(self, clip1, clip2, duration):
        """플래시(화이트아웃) 트랜지션"""
        # 화이트 프레임 생성
        w, h = clip1.size
        white_frame = np.ones((h, w, 3), dtype=np.uint8) * 255
        white_clip = ImageClip(white_frame).set_duration(duration * 0.3)
        
        clip1_fade = clip1.fadeout(duration * 0.3)
        clip2_fade = clip2.fadein(duration * 0.3)
        
        return concatenate_videoclips([clip1_fade, white_clip, clip2_fade])
    
    def _wipe(self, clip1, clip2, direction, duration):
        """와이프 트랜지션"""
        w, h = clip1.size
        
        def make_frame(t):
            progress = t / duration
            
            if t < clip1.duration - duration:
                return clip1.get_frame(t)
            
            local_t = t - (clip1.duration - duration)
            local_progress = min(1.0, local_t / duration)
            
            frame1 = clip1.get_frame(min(t, clip1.duration - 0.01))
            frame2 = clip2.get_frame(max(0, t - clip1.duration + duration))
            
            result = frame1.copy()
            
            if direction == 'left':
                split_x = int(w * (1 - local_progress))
                result[:, :w-split_x] = frame2[:, split_x:]
            else:
                split_x = int(w * local_progress)
                result[:, split_x:] = frame2[:, :w-split_x]
            
            return result
        
        total_duration = clip1.duration + clip2.duration - duration
        return VideoFileClip(make_frame, duration=total_duration)
    
    def _zoom_transition(self, clip1, clip2, duration):
        """줌 트랜지션"""
        # clip1이 줌아웃 되면서 clip2가 줌인 되며 나타남
        clip1_zoom = clip1.resize(lambda t: max(0.1, 1 - (t / clip1.duration) * 0.3) 
                                   if t > clip1.duration - duration else 1)
        clip1_zoom = clip1_zoom.fadeout(duration)
        clip2 = clip2.fadein(duration)
        clip2 = clip2.set_start(clip1.duration - duration)
        
        return CompositeVideoClip([clip1_zoom, clip2])


transition_engine = TransitionEngine()
print("✅ 트랜지션 엔진 초기화 완료!")

## 8. 미디어 분석기

In [ ]:
class MediaAnalyzer:
    """업로드된 미디어 분석"""
    
    def __init__(self):
        self.face_detector = face_detector
    
    def analyze_media(self, filepath: str) -> MediaInfo:
        """미디어 파일 분석"""
        ext = os.path.splitext(filepath)[1].lower()
        
        if ext in ['.jpg', '.jpeg', '.png', '.bmp', '.webp']:
            return self._analyze_image(filepath)
        elif ext in ['.mp4', '.avi', '.mov', '.mkv', '.webm']:
            return self._analyze_video(filepath)
        elif ext in ['.mp3', '.wav', '.ogg', '.m4a', '.flac']:
            return self._analyze_audio(filepath)
        else:
            raise ValueError(f"지원하지 않는 파일 형식: {ext}")
    
    def _analyze_image(self, filepath: str) -> MediaInfo:
        """이미지 분석"""
        print(f"🖼️ 이미지 분석 중: {os.path.basename(filepath)}")
        
        img = cv2.imread(filepath)
        h, w = img.shape[:2]
        
        # 얼굴 감지
        faces = self.face_detector.detect_faces(img)
        
        # 주요 색상 추출
        dominant_colors = self._extract_colors(img)
        
        print(f"  - 크기: {w}x{h}")
        print(f"  - 감지된 얼굴: {len(faces)}개")
        
        return MediaInfo(
            filepath=filepath,
            media_type='image',
            duration=Config.DEFAULT_IMAGE_DURATION,
            width=w,
            height=h,
            faces=faces,
            has_motion=False,
            dominant_colors=dominant_colors
        )
    
    def _analyze_video(self, filepath: str) -> MediaInfo:
        """비디오 분석"""
        print(f"🎬 비디오 분석 중: {os.path.basename(filepath)}")
        
        cap = cv2.VideoCapture(filepath)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = frame_count / fps if fps > 0 else 0
        
        # 첫 프레임에서 얼굴 감지
        ret, frame = cap.read()
        faces = self.face_detector.detect_faces(frame) if ret else []
        
        # 모션 감지 (첫 프레임과 중간 프레임 비교)
        has_motion = False
        if frame_count > 30:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_count // 2)
            ret2, frame2 = cap.read()
            if ret and ret2:
                diff = cv2.absdiff(frame, frame2)
                has_motion = np.mean(diff) > 10
        
        cap.release()
        
        print(f"  - 크기: {w}x{h}")
        print(f"  - 길이: {duration:.2f}초")
        print(f"  - 감지된 얼굴: {len(faces)}개")
        print(f"  - 모션: {'있음' if has_motion else '없음'}")
        
        return MediaInfo(
            filepath=filepath,
            media_type='video',
            duration=duration,
            width=w,
            height=h,
            faces=faces,
            has_motion=has_motion
        )
    
    def _analyze_audio(self, filepath: str) -> MediaInfo:
        """오디오 분석"""
        print(f"🎵 오디오 분석 중: {os.path.basename(filepath)}")
        
        y, sr = librosa.load(filepath)
        duration = librosa.get_duration(y=y, sr=sr)
        
        print(f"  - 길이: {duration:.2f}초")
        
        return MediaInfo(
            filepath=filepath,
            media_type='audio',
            duration=duration
        )
    
    def _extract_colors(self, img, k=3):
        """주요 색상 추출"""
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pixels = img_rgb.reshape(-1, 3).astype(np.float32)
        
        # K-means로 주요 색상 추출
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
        _, labels, centers = cv2.kmeans(pixels, k, None, criteria, 3, cv2.KMEANS_RANDOM_CENTERS)
        
        return [tuple(map(int, c)) for c in centers]


media_analyzer = MediaAnalyzer()
print("✅ 미디어 분석기 초기화 완료!")

## 9. 자동 편집 AI 엔진

In [ ]:
class AutoEditorAI:
    """AI 기반 자동 편집 결정 엔진"""
    
    def __init__(self):
        self.camera_engine = camera_engine
        self.transition_engine = transition_engine
    
    def create_edit_decisions(self, media_list: List[MediaInfo], 
                               beat_info: BeatInfo) -> List[EditDecision]:
        """미디어와 음악을 분석하여 편집 결정 생성"""
        decisions = []
        current_time = 0.0
        media_index = 0
        beat_index = 0
        strong_beat_index = 0
        
        print("\n🎬 편집 계획 생성 중...")
        
        while current_time < beat_info.total_duration and media_index < len(media_list) * 3:
            media = media_list[media_index % len(media_list)]
            
            # 클립 길이 결정 (비트에 맞춤)
            clip_duration = self._calculate_clip_duration(
                current_time, beat_info, media
            )
            
            # 남은 시간 체크
            if current_time + clip_duration > beat_info.total_duration:
                clip_duration = beat_info.total_duration - current_time
            
            if clip_duration < 0.5:
                break
            
            # 카메라 기법 선택
            technique = self._select_camera_technique(
                media, current_time, beat_info, strong_beat_index
            )
            
            # 트랜지션 선택
            transition_in, transition_out = self._select_transitions(
                current_time, beat_info, len(decisions)
            )
            
            # 줌 타겟 설정
            zoom_target = media.faces[0] if media.faces else None
            
            decision = EditDecision(
                media_index=media_index % len(media_list),
                start_time=current_time,
                end_time=current_time + clip_duration,
                camera_technique=technique,
                transition_in=transition_in,
                transition_out=transition_out,
                zoom_target=zoom_target
            )
            decisions.append(decision)
            
            print(f"  [{len(decisions)}] {current_time:.1f}s-{current_time+clip_duration:.1f}s: "
                  f"{technique.value} | {os.path.basename(media.filepath)}")
            
            current_time += clip_duration - Config.TRANSITION_DURATION
            media_index += 1
            
            # 강한 비트 인덱스 업데이트
            while (strong_beat_index < len(beat_info.strong_beats) and 
                   beat_info.strong_beats[strong_beat_index] < current_time):
                strong_beat_index += 1
        
        return decisions
    
    def _calculate_clip_duration(self, current_time: float, 
                                  beat_info: BeatInfo, media: MediaInfo) -> float:
        """비트에 맞춘 클립 길이 계산"""
        # 다음 강한 비트까지의 시간 찾기
        next_strong_beat = None
        for beat_time in beat_info.strong_beats:
            if beat_time > current_time + Config.MIN_CLIP_DURATION:
                next_strong_beat = beat_time
                break
        
        if next_strong_beat:
            duration = next_strong_beat - current_time
        else:
            # 일반 비트 기준으로 2-4비트 길이
            beat_interval = 60.0 / beat_info.tempo
            num_beats = random.choice([2, 3, 4])
            duration = beat_interval * num_beats
        
        # 범위 제한
        duration = max(Config.MIN_CLIP_DURATION, 
                      min(Config.MAX_CLIP_DURATION, duration))
        
        # 비디오인 경우 원본 길이 고려
        if media.media_type == 'video':
            duration = min(duration, media.duration)
        
        return duration
    
    def _select_camera_technique(self, media: MediaInfo, current_time: float,
                                  beat_info: BeatInfo, strong_beat_index: int) -> CameraTechnique:
        """상황에 맞는 카메라 기법 선택"""
        
        has_face = len(media.faces) > 0 if media.faces else False
        
        # 강한 비트 근처인지 확인
        is_strong_beat = any(
            abs(bt - current_time) < 0.3 
            for bt in beat_info.strong_beats
        )
        
        if media.media_type == 'image':
            if has_face:
                # 얼굴이 있는 이미지: 줌인/줌아웃 강조
                if is_strong_beat:
                    return CameraTechnique.FACE_TRACK
                else:
                    return random.choice([
                        CameraTechnique.ZOOM_IN,
                        CameraTechnique.ZOOM_OUT,
                        CameraTechnique.KEN_BURNS
                    ])
            else:
                # 풍경 등: 패닝이나 켄 번스
                return random.choice([
                    CameraTechnique.KEN_BURNS,
                    CameraTechnique.PAN_LEFT,
                    CameraTechnique.PAN_RIGHT,
                    CameraTechnique.TILT_UP,
                    CameraTechnique.DOLLY_IN
                ])
        
        else:  # video
            if has_face and is_strong_beat:
                return CameraTechnique.FACE_TRACK
            elif has_face:
                return random.choice([
                    CameraTechnique.ZOOM_IN,
                    CameraTechnique.ZOOM_OUT,
                    CameraTechnique.STATIC
                ])
            elif media.has_motion:
                return random.choice([
                    CameraTechnique.STATIC,
                    CameraTechnique.DOLLY_IN,
                    CameraTechnique.SHAKE
                ])
            else:
                return random.choice([
                    CameraTechnique.DOLLY_IN,
                    CameraTechnique.PAN_LEFT,
                    CameraTechnique.PAN_RIGHT
                ])
    
    def _select_transitions(self, current_time: float, beat_info: BeatInfo,
                            clip_index: int) -> Tuple[TransitionType, TransitionType]:
        """트랜지션 선택"""
        
        # 강한 비트에서는 임팩트 있는 트랜지션
        is_strong_beat = any(
            abs(bt - current_time) < 0.5 
            for bt in beat_info.strong_beats
        )
        
        if clip_index == 0:
            transition_in = TransitionType.FADE
        elif is_strong_beat:
            transition_in = random.choice([
                TransitionType.CUT,
                TransitionType.FLASH,
                TransitionType.ZOOM_TRANSITION
            ])
        else:
            transition_in = random.choice([
                TransitionType.CUT,
                TransitionType.CROSSFADE,
                TransitionType.WIPE_LEFT,
                TransitionType.WIPE_RIGHT
            ])
        
        transition_out = TransitionType.CUT  # 기본값, 다음 클립의 in이 결정
        
        return transition_in, transition_out


auto_editor = AutoEditorAI()
print("✅ 자동 편집 AI 엔진 초기화 완료!")

## 10. 비디오 렌더러

In [ ]:
class VideoRenderer:
    """최종 비디오 렌더링"""
    
    def __init__(self):
        self.camera_engine = camera_engine
        self.transition_engine = transition_engine
    
    def render(self, media_list: List[MediaInfo], decisions: List[EditDecision],
               audio_path: str, output_path: str = "output_video.mp4"):
        """편집 결정에 따라 최종 비디오 렌더링"""
        print("\n🎬 비디오 렌더링 시작...")
        
        clips = []
        
        for i, decision in enumerate(decisions):
            print(f"  클립 {i+1}/{len(decisions)} 처리 중...")
            
            media = media_list[decision.media_index]
            clip_duration = decision.end_time - decision.start_time
            
            # 클립 생성
            if media.media_type == 'image':
                clip = self._create_image_clip(media, clip_duration)
            else:
                clip = self._create_video_clip(media, clip_duration)
            
            # 출력 해상도에 맞게 조정
            clip = self._fit_to_output(clip)
            
            # 카메라 효과 적용
            clip = self.camera_engine.apply_effect(
                clip, 
                decision.camera_technique,
                decision.zoom_target
            )
            
            clips.append({
                'clip': clip,
                'decision': decision
            })
        
        # 트랜지션 적용하며 연결
        print("  트랜지션 적용 중...")
        final_clip = self._apply_transitions(clips)
        
        # 오디오 추가
        print("  오디오 추가 중...")
        audio = AudioFileClip(audio_path)
        final_clip = final_clip.set_audio(audio.subclip(0, final_clip.duration))
        
        # 렌더링
        print(f"  파일 저장 중: {output_path}")
        final_clip.write_videofile(
            output_path,
            fps=Config.OUTPUT_FPS,
            codec='libx264',
            audio_codec='aac',
            preset='medium',
            threads=4
        )
        
        # 정리
        final_clip.close()
        audio.close()
        
        print(f"\n✅ 렌더링 완료: {output_path}")
        return output_path
    
    def _create_image_clip(self, media: MediaInfo, duration: float):
        """이미지에서 클립 생성"""
        return ImageClip(media.filepath).set_duration(duration)
    
    def _create_video_clip(self, media: MediaInfo, duration: float):
        """비디오에서 클립 생성"""
        clip = VideoFileClip(media.filepath)
        
        # 필요한 길이만큼 자르기
        if clip.duration > duration:
            # 중간 부분 사용
            start = (clip.duration - duration) / 2
            clip = clip.subclip(start, start + duration)
        elif clip.duration < duration:
            # 루프
            clip = clip.loop(duration=duration)
        
        return clip
    
    def _fit_to_output(self, clip):
        """출력 해상도에 맞게 조정"""
        target_w = Config.OUTPUT_WIDTH
        target_h = Config.OUTPUT_HEIGHT
        target_ratio = target_w / target_h
        
        clip_w, clip_h = clip.size
        clip_ratio = clip_w / clip_h
        
        if clip_ratio > target_ratio:
            # 원본이 더 넓음 - 높이에 맞추고 좌우 크롭
            new_h = target_h
            new_w = int(clip_ratio * target_h)
            clip = clip.resize(height=new_h)
            x_center = new_w // 2
            clip = clip.crop(x1=x_center - target_w//2, 
                            x2=x_center + target_w//2,
                            y1=0, y2=target_h)
        else:
            # 원본이 더 좁음 - 너비에 맞추고 상하 크롭
            new_w = target_w
            new_h = int(target_w / clip_ratio)
            clip = clip.resize(width=new_w)
            y_center = new_h // 2
            clip = clip.crop(x1=0, x2=target_w,
                            y1=y_center - target_h//2,
                            y2=y_center + target_h//2)
        
        return clip
    
    def _apply_transitions(self, clips_data: List[Dict]):
        """트랜지션을 적용하며 클립 연결"""
        if not clips_data:
            return None
        
        if len(clips_data) == 1:
            return clips_data[0]['clip']
        
        result = clips_data[0]['clip']
        
        for i in range(1, len(clips_data)):
            current_clip = clips_data[i]['clip']
            transition_type = clips_data[i]['decision'].transition_in
            
            result = self.transition_engine.apply_transition(
                result, current_clip, transition_type, Config.TRANSITION_DURATION
            )
        
        return result


renderer = VideoRenderer()
print("✅ 비디오 렌더러 초기화 완료!")

## 11. 메인 컨트롤러

In [ ]:
class AutoVideoEditor:
    """자동 비디오 편집 메인 컨트롤러"""
    
    def __init__(self):
        self.media_analyzer = media_analyzer
        self.music_analyzer = music_analyzer
        self.auto_editor = auto_editor
        self.renderer = renderer
        
        self.media_list = []
        self.audio_path = None
        self.beat_info = None
        
        # 작업 디렉토리 생성
        os.makedirs('uploads', exist_ok=True)
        os.makedirs('output', exist_ok=True)
    
    def upload_files(self):
        """파일 업로드"""
        print("📁 파일을 업로드하세요 (이미지, 비디오, 음악)")
        print("   지원 형식:")
        print("   - 이미지: jpg, jpeg, png, bmp, webp")
        print("   - 비디오: mp4, avi, mov, mkv, webm")
        print("   - 오디오: mp3, wav, ogg, m4a, flac")
        print()
        
        uploaded = files.upload()
        
        for filename, content in uploaded.items():
            filepath = os.path.join('uploads', filename)
            with open(filepath, 'wb') as f:
                f.write(content)
            print(f"  ✅ 저장됨: {filename}")
        
        return list(uploaded.keys())
    
    def analyze_uploads(self):
        """업로드된 파일 분석"""
        print("\n" + "="*50)
        print("📊 미디어 분석 시작")
        print("="*50)
        
        upload_dir = 'uploads'
        files_list = os.listdir(upload_dir)
        
        self.media_list = []
        self.audio_path = None
        
        for filename in files_list:
            filepath = os.path.join(upload_dir, filename)
            media_info = self.media_analyzer.analyze_media(filepath)
            
            if media_info.media_type == 'audio':
                self.audio_path = filepath
                # 음악 비트 분석
                self.beat_info = self.music_analyzer.analyze(filepath)
            else:
                self.media_list.append(media_info)
        
        if not self.audio_path:
            raise ValueError("❌ 오디오 파일이 필요합니다!")
        
        if not self.media_list:
            raise ValueError("❌ 이미지 또는 비디오 파일이 필요합니다!")
        
        print("\n" + "="*50)
        print("📋 분석 결과 요약")
        print("="*50)
        print(f"  - 이미지/비디오: {len(self.media_list)}개")
        print(f"  - 음악 길이: {self.beat_info.total_duration:.1f}초")
        print(f"  - 템포: {self.beat_info.tempo:.1f} BPM")
        
        total_faces = sum(len(m.faces) if m.faces else 0 for m in self.media_list)
        print(f"  - 총 감지된 얼굴: {total_faces}개")
    
    def create_video(self, output_filename: str = "auto_edited_video.mp4"):
        """자동 편집 비디오 생성"""
        print("\n" + "="*50)
        print("🎬 자동 편집 시작")
        print("="*50)
        
        # 편집 결정 생성
        decisions = self.auto_editor.create_edit_decisions(
            self.media_list, self.beat_info
        )
        
        # 렌더링
        output_path = os.path.join('output', output_filename)
        self.renderer.render(
            self.media_list, decisions, self.audio_path, output_path
        )
        
        return output_path
    
    def run(self):
        """전체 파이프라인 실행"""
        print("\n" + "🎬"*20)
        print("    AI 자동 비디오 편집기")
        print("🎬"*20 + "\n")
        
        # 1. 파일 업로드
        self.upload_files()
        
        # 2. 분석
        self.analyze_uploads()
        
        # 3. 비디오 생성
        output_path = self.create_video()
        
        # 4. 다운로드 제공
        print("\n" + "="*50)
        print("📥 비디오 다운로드")
        print("="*50)
        files.download(output_path)
        
        return output_path


# 전역 에디터 인스턴스
editor = AutoVideoEditor()
print("✅ 자동 비디오 편집기 준비 완료!")

## 12. 실행!

In [ ]:
# 자동 편집기 실행
output_video = editor.run()

## 13. 결과 미리보기 (선택사항)

In [ ]:
# 결과 비디오 미리보기 (Colab에서 재생)
from IPython.display import HTML
from base64 import b64encode

def show_video(video_path):
    """비디오를 노트북에서 재생"""
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f"""
    <video width="800" controls>
        <source src="{data_url}" type="video/mp4">
    </video>
    """)

# 비디오 표시
show_video(output_video)

## 14. 설정 커스터마이징 (선택사항)

In [ ]:
# 설정을 변경하고 다시 실행하려면:

# 출력 해상도 변경
Config.OUTPUT_WIDTH = 1920   # 4K: 3840
Config.OUTPUT_HEIGHT = 1080  # 4K: 2160

# 클립 길이 조정
Config.MIN_CLIP_DURATION = 1.5  # 최소 클립 길이 (초)
Config.MAX_CLIP_DURATION = 8.0  # 최대 클립 길이 (초)

# 줌 강도 조정
Config.ZOOM_INTENSITY = 1.3      # 일반 줌 강도 (1.0 = 변화없음)
Config.FACE_ZOOM_INTENSITY = 1.5  # 얼굴 줌 강도

# 트랜지션 길이
Config.TRANSITION_DURATION = 0.5  # 초

print("✅ 설정이 업데이트되었습니다!")
print("다시 실행하려면 위의 '12. 실행!' 셀을 다시 실행하세요.")